## Tools
Connect claude with external world, for example, asking about sanfranciso time

![alt text](images/tools.png)

![alt text](images/tool_weather_example.png)

## Writing a tool function
- Very important to validate input parameters to that function and return good error messages
- For tools use a JSON schema well structured, a good json schema uses a good description
- A tool is composed of python function + schema

![alt text](images/toool_good_descriptions.png)

In [ ]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
}

## Notice the pattern: 
function_name ->
function_name_schema

## Better type checking
best schemas using anthropic official libraries

In [ ]:
from anthropic.types import ToolParam

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    # ... rest of schema
})

## Tool enabled API calls

In [ ]:
messages = []
messages.append({
    "role": "user",
    "content": "What is the exact time, formatted as HH:MM:SS?"
})

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

Notice:     
tools=[get_current_datetime_schema],


## Multiblock messages
When claude decides to use a tool, it creates a multiblock message

![alt text](images/tools_multiblock_messages.png)


Notice: Text block & ToolUse Block

## Tool usage process
The tool usage process follows this pattern:

- Send user message with tool schema to Claude
- Receive assistant message with text block and tool use block
- Extract tool information and execute the actual function
- Send tool result back to Claude along with complete conversation history
- Receive final response from Claude

## Handling tool use blocks
When claude responds with a tool use block, we extract it and run the tool

In [ ]:
get_current_datetime(**response.content[1].input)

## Returning the tool result block
After running the tool function, we need to send th results back to Claude using a tool result block. This block goes inside a user message and tells claude what happened when you executed the tool


![alt text](images/tools_tool_result_block.png)

The tool result block has several important properties:

- tool_use_id - Must match the id of the ToolUse block that this ToolResult corresponds to
- content - Output from running your tool, serialized as a string
- is_error - True if an error occurred

## Handling Multiple Tool Calls

Example, user asks "What's 10 + 10 and what's 30 + 30?", Claude might respond with two separate ToolUse blocks.

![alt text](images/tools_many_tool_usage.png)

## Building the followup request
Your folllwup request to claude must include the complete coversation history plus the new tool result. Here is the strcuture:

In [ ]:
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": response.content[1].id,
        "content": "15:04:22",
        "is_error": False
    }]
})

Notice: We attach the tool usage to assitant message, and the tool result as user message

## Guess what?

The complete message history now contains:

- Original user message
- Assistant message with tool use block
- User message with tool result block


![alt text](images/tools_complete_lifecycle.png)


## Multiturn conversation
For example: if a user asks "What day is 103 days from today?", Claude needs to first get the current date, then add 103 days to it.

![alt-text](images/tool_multiturn.png)

## Conversation Loop
To handle this pattern, you need a conversation loop that continues until claude stops requesting tools

In [ ]:
def run_conversation(messages):
    while True:
        response = chat(messages)

        add_assistant_message(messages, response)

        # Pseudo code
        if response isn't asking for a tool:
            break

        tool_result_blocks = run_tools(response)
        add_user_message(messages, tool_result_blocks)
        
    return messages

## So what this means?
It means that we need to refactor helper functions to handle multiple messages blocks properly.
Before this, we always consider that we were always working with plain text, but now we are using 
- ToolUse 
- ToolResult blocks

In [ ]:
from anthropic.types import Message

def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)

And now that we are using tools, we need to modigy chat function to accept a list of tools and return the full text instead of just text

In [ ]:
def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    
    if tools:
        params["tools"] = tools
        
    if system:
        params["system"] = system
        
    message = client.messages.create(**params)
    return message

## Extracting text form messages
Now that we are returning full message objects, we need a helper to get the text

In [ ]:
def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

## Improvements

- Flexible message handling - Your helper functions can now work with different message formats
- Tool support in chat - The chat function can receive and pass through tool schemas
- Full message returns - You get complete message objects instead of just text, preserving all blocks
- Text extraction utility - Easy way to get readable text from complex messages

## Handle multiple tool calls
Claude can request multiple tools in a single reponse, for that, we use a robust and scalable
- run tools
- run tool

In [ ]:
import json


def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

## Guess what? 
Now we have multi-turn conversation with multiple tool usage (requested by claude)

The complete multi-turn conversation works like this:

- Send user message to Claude with available tools
- Claude responds with text and/or tool requests
- Execute all requested tools and create result blocks
- Send tool results back as a user message
- Repeat until Claude provides a final answer

## Adding more tools
Remember to append in run_tool, attach function and schema!

pattern:

- Create the tool function implementation
- Define the tool schema
- Add the schema to the tools list in run_conversation
- Add a case for the tool in run_tool

## Streaming tool usage

You can have streaming when you use tools, you must enable it

![alt text](images/tools_with_streaming.png)

# Note: 
With streaming enabled is very important to notice that api does the following:

In [ ]:
## Consider the following response

{
  "abstract": "This paper presents a novel...",
  "meta": {
    "word_count": 847,
    "review": "This paper introduces QuanNet..."
  }
}

- Wait until the entire abstract value is complete
- Validate that key-value pair against your schema
- Send all the buffered chunks for abstract at once
- Repeat the process for the meta object

## You need real realtime?
## Fine graned tool calling
It is used to get each chunk of generated JSON as fast as posible.
Fine grained tools sends froups of chunks BUT WHITHOUT WAITING FOR A FULL TOP LEVEL KEY TO BE CREATED


## The tradeoff: 
JSON Validation is disabled!
Your code should handle invalid tool inputs

In [ ]:
## Enable it
run_conversation(
    messages, 
    tools=[save_article_schema], 
    fine_grained=True
)

## When to use Fine-Grained tool calling?
- You need to show users real-time progress on tool argument generation
- You want to start processing partial tool results as quickly as possible
- The buffering delays negatively impact your user experience
- You're comfortable implementing robust JSON error handling

## Text editor tool
Practical Example
Let's see the text editor tool in action. When you ask Claude to work with files, it will use the tool to read, modify, and create files as needed.

For example, if you ask Claude to "Open the ./main.py file and summarize its contents", Claude will:

Use the text editor tool to view the file
Read the contents
Provide you with a summary

## Use cases

- You're building applications that need to programmatically edit files
- You're working in environments without access to full-featured code editors
- You want to integrate file editing capabilities directly into your Claude-powered applications